In [4]:
# !pip install numpy


In [8]:
# !pip install matplotlib
!pip install openpyxl


  Obtaining dependency information for openpyxl from https://files.pythonhosted.org/packages/c0/da/977ded879c29cbd04de313843e76868e6e13408a94ed6b987245dc7c8506/openpyxl-3.1.5-py2.py3-none-any.whl.metadata
  Obtaining dependency information for et-xmlfile from https://files.pythonhosted.org/packages/c1/8b/5fe2cc11fee489817272089c4203e679c63b570a5aaeb18d852ae3cbba6a/et_xmlfile-2.0.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/250.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/250.9 kB ? eta -:--:--
   ----------- --------------------------- 71.7/250.9 kB 660.6 kB/s eta 0:00:01
   ------------------------ --------------- 153.6/250.9 kB 1.0 MB/s eta 0:00:01
   ---------------------------------------- 250.9/250.9 kB 1.5 MB/s eta 0:00:00


In [17]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import subprocess
import sys

In [11]:
# File path for the Excel workbook
file_path = "Ponudba.xlsx"  # Make sure this file is in the same directory or specify the full path

try:
    # Load all sheets from the Excel file
    sheets_dict = pd.read_excel(file_path, sheet_name=None)  # Load all sheets

    print("Available products and prices by category:")

    # Loop through each sheet (category) and print products with prices
    for category, df in sheets_dict.items():
        print(f"\nCategory: {category}")
        for item, price in zip(df.iloc[:, 0], df.iloc[:, 1]):  # Iterate over item names and prices
            print(f"  {item}: ${price}")
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")


Available products and prices by category:

Category: Sodi
  Malvazija 25: $75
  Refošk 25: $100
  Malvazija 50: $150
  Refošk 50: $200

Category: Butelke
  Malvazija 23: $9
  Refošk 23: $9
  Malvazija 16: $6
  Refošk 16: $5
  Shiraz 18: $5
  Muškat: $15
  Merlot: $2

Category: Drugo
  Malvazija 1L: $3
  Refošk 1L: $3
  Olje: $3


In [ ]:
import subprocess
import sys

# Function to install required packages if not already installed
def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Install required libraries
try:
    import pandas as pd
except ImportError:
    install('pandas')

try:
    import tkinter as tk
    from tkinter import simpledialog, messagebox
except ImportError:
    install('tkinter')  # tk is usually bundled with Python, but if it's not, we use install

from datetime import datetime

# Read the Excel file with multiple sheets
file_path = "Ponudba.xlsx"  # Ensure this file is in the same directory or specify the full path

try:
    # Load all sheets from the Excel file
    sheets_dict = pd.read_excel(file_path, sheet_name=None)  # Load all sheets

    # Flatten the items into a single dictionary (across all sheets)
    items = {}
    for category, df in sheets_dict.items():
        items.update(dict(zip(df.iloc[:, 0], df.iloc[:, 1])))  # Add item names and prices from each sheet

except FileNotFoundError:
    messagebox.showerror("Error", f"Error: The file '{file_path}' was not found.")
except Exception as e:
    messagebox.showerror("Error", f"An error occurred: {e}")

# Function to save receipt as .txt file
def save_receipt_as_txt(customer_name, order_data, is_receipt=True):
    # Get current date and time for file naming
    now = datetime.now()
    suffix = "_brezr" if not is_receipt else ""
    filename = f"{customer_name}_{now.strftime('%d%m%y_%H%M')}{suffix}.txt"

    # Open the file in write mode
    with open(filename, "w") as f:
        # Write the customer name
        f.write(f"Customer: {customer_name}\n\n")

        # Write the itemized list
        total_price = 0
        for item, qty in order_data.items():
            price = items.get(item, 0)
            line_total = price * qty
            total_price += line_total
            f.write(f"{item} x{qty} = ${line_total}\n")

        # Write the total price
        f.write(f"\nTotal: ${total_price}\n")

    # Show a success message
    messagebox.showinfo("Success", f"Order saved as {filename}")

# Function to log the receipt data into Excel
def log_receipt_in_excel(customer_name, order_data, is_receipt=True):
    # Get current date and time for logging
    now = datetime.now()
    year = now.year
    date_time = now.strftime('%d-%m-%Y %H:%M')

    # Determine the sheet name based on receipt status
    sheet_name = str(year) if is_receipt else "Brez_računa"

    # Try to load the existing Excel file or create a new one
    try:
        # If the file exists, load it
        if pd.io.common.file_exists("Računi.xlsx"):
            # Read the Excel file
            with pd.ExcelFile("Računi.xlsx") as xls:
                # Check if the sheet for the current year or "Brez_računa" exists
                if sheet_name in xls.sheet_names:
                    df = pd.read_excel(xls, sheet_name=sheet_name)
                else:
                    # If no sheet for the current year, create a new one
                    df = pd.DataFrame(columns=["Customer", "Date and Time"] + list(items.keys()) + ["Total Price"])
        else:
            # If the file doesn't exist, create a new one
            df = pd.DataFrame(columns=["Customer", "Date and Time"] + list(items.keys()) + ["Total Price"])

        # Prepare the row data (customer, date, time, item quantities, and total price)
        row = [customer_name, date_time]
        total_price = 0
        for item in items.keys():
            qty = order_data.get(item, 0)
            price = items[item]
            row.append(qty)
            total_price += price * qty
        row.append(total_price)

        # Append the new order data to the DataFrame
        df.loc[len(df)] = row

        # Write the updated DataFrame to the Excel file (sheet for the current year)
        with pd.ExcelWriter("Računi.xlsx", engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

    except Exception as e:
        messagebox.showerror("Error", f"An error occurred while logging the receipt: {e}")
        return

    # Show a success message
    messagebox.showinfo("Success", "Order logged in Excel successfully.")

# Function to collect order details
def collect_order():
    # Create the root window for the GUI
    root = tk.Tk()
    root.title("Customer Order")

    # Ask for customer name
    customer_name = simpledialog.askstring("Customer Name", "Enter the customer name:")

    if not customer_name:
        messagebox.showerror("Error", "Customer name is required!")
        return

    # Ask if a receipt is needed
    is_receipt = messagebox.askyesno("Receipt Needed", "Do you need a receipt?")

    # Create a dictionary to store the quantities entered for each item
    order_data = {}

    # Create a frame for the items and quantities
    frame = tk.Frame(root)
    frame.pack(padx=10, pady=10)

    # Label for each item with an input field for quantity
    for idx, (item, price) in enumerate(items.items()):
        tk.Label(frame, text=f"{item} (${price})").grid(row=idx, column=0, sticky="w", padx=10)
        quantity_entry = tk.Entry(frame)
        quantity_entry.grid(row=idx, column=1)
        order_data[item] = quantity_entry

    # Submit button to process the order
    def submit_order():
        quantities = {}
        for item, entry in order_data.items():
            try:
                qty = int(entry.get())  # Get the quantity as an integer
                if qty > 0:
                    quantities[item] = qty
            except ValueError:
                continue  # Ignore invalid entries (non-numeric)
        
        # Show confirmation
        if quantities:
            message = f"Order for {customer_name}:\n"
            total_price = 0
            for item, qty in quantities.items():
                price = items[item]
                total_price += price * qty
                message += f"{item} x{qty} = ${price * qty}\n"
            message += f"\nTotal: ${total_price}"
            messagebox.showinfo("Order Summary", message)
            
            # Save the receipt as .txt
            save_receipt_as_txt(customer_name, quantities, is_receipt)
            
            # Log the receipt in Excel
            log_receipt_in_excel(customer_name, quantities, is_receipt)
        else:
            messagebox.showerror("Error", "No valid quantities entered!")

        # Close the window after submitting the order
        root.quit()

    # Submit button
    submit_button = tk.Button(root, text="Submit Order", command=submit_order)
    submit_button.pack(pady=10)

    # Start the GUI event loop
    root.mainloop()

# Call the function to collect the order
collect_order()


: 